In [1]:
!pip install -q transformers sentence-transformers faiss-cpu torch sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 27.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ==========================================================
# 1. Knowledge Base
# ==========================================================

documents = [

"""
Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.
""",

"""
Large Language Models are transformer-based models trained on massive
text datasets. They are used for text generation, summarization,
translation, question answering and conversational AI.
""",

"""
Retrieval-Augmented Generation combines information retrieval with
text generation. It retrieves relevant documents from an external
knowledge base and gives them to a language model as context.
""",

"""
Vector databases store high-dimensional embeddings and perform
similarity searches. Examples include FAISS, ChromaDB,
Pinecone, Weaviate and Milvus.
""",

"""
Prompt engineering is the process of designing clear instructions
that guide a language model to produce accurate and useful responses.
Common techniques include zero-shot, few-shot and role-based prompting.
""",

"""
Fine-tuning adapts a pretrained language model to a specific domain
or task by training it further using a smaller domain-specific dataset.
"""

]

# ==========================================================
# 2. Load Embedding Model
# ==========================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# ==========================================================
# 3. Create Document Embeddings
# ==========================================================

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(document_embeddings)


# ==========================================================
# 4. Create FAISS Index
# ==========================================================

embedding_dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatIP(embedding_dimension)

vector_database.add(document_embeddings)


# ==========================================================
# 5. Load FLAN-T5
# ==========================================================

print("Loading FLAN-T5 model...")

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# ==========================================================
# 6. Retrieval Function
# ==========================================================

def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    similarity_scores, document_indices = vector_database.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for index, score in zip(document_indices[0], similarity_scores[0]):

        retrieved_documents.append({
            "document": documents[index].strip(),
            "score": float(score)
        })

    return retrieved_documents


# ==========================================================
# 7. Generate Answer
# ==========================================================

def generate_answer(query, retrieved_documents):

    context = "\n\n".join(
        item["document"] for item in retrieved_documents
    )

    prompt = f"""
Answer the question using ONLY the information given below.

Context:
{context}

Question:
{query}

Instructions:
1. Give a short answer.
2. Use only the context.
3. If the answer is not present, say:
"The answer is not available in the knowledge base."

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


# ==========================================================
# 8. Main Program
# ==========================================================

print("="*60)
print("RETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("="*60)

user_query = input("\nEnter your question: ")

retrieved_results = retrieve_documents(
    user_query,
    top_k=2
)

answer = generate_answer(
    user_query,
    retrieved_results
)


# ==========================================================
# 9. Display Retrieved Documents
# ==========================================================

print("\n")
print("="*60)
print("RETRIEVED DOCUMENTS")
print("="*60)

for i, item in enumerate(retrieved_results, start=1):

    print(f"\nDocument {i}")
    print("-"*40)

    print(item["document"])

    print(f"\nSimilarity Score : {item['score']:.4f}")


# ==========================================================
# 10. Display Final Answer
# ==========================================================

print("\n")
print("="*60)
print("GENERATED ANSWER")
print("="*60)

print(answer)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

RETRIEVAL-AUGMENTED GENERATION SYSTEM

Enter your question: What is Retrieval-Augmented Generation?


RETRIEVED DOCUMENTS

Document 1
----------------------------------------
Retrieval-Augmented Generation combines information retrieval with
text generation. It retrieves relevant documents from an external
knowledge base and gives them to a language model as context.

Similarity Score : 0.6933

Document 2
----------------------------------------
Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.

Similarity Score : 0.3435


GENERATED ANSWER
combines information retrieval with text generation
